In [1]:
from embedder import Embedder

embedder = Embedder()

nn_query = "How does approximate nearest neighbor search work?"
nn_query_embedding = embedder.encode(nn_query)

print(f"The first value is {nn_query_embedding[0]}")

The first value is -0.02058203437252893


### Loading the data & Q2. Cosine similarity

In [2]:
from gitsource import GithubRepositoryDataReader

github_reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

github_docs = [file.parse() for file in github_reader.read()]

print(f"Total of documents: {len(github_docs)}")
print("----------------------")
print(f"Document structure: \n {list(github_docs[0].keys())}")
print("----------------------")

Total of documents: 72
----------------------
Document structure: 
 ['content', 'filename']
----------------------


In [3]:
target_document = ""
for doc in github_docs:
    if "07-sqlitesearch-vector.md" in doc["filename"]:
        target_document = doc["content"]
        break

target_doc_embedding = embedder.encode(target_document)
cosine_similarity = target_doc_embedding.dot(nn_query_embedding)

print(f"The cosine similarity is {cosine_similarity}")

The cosine similarity is 0.36107027225589694


### Q3. Chunking and search by hand

In [4]:
from gitsource import chunk_documents
import numpy as np
from tqdm.auto import tqdm

doc_chunks = chunk_documents(github_docs, size=2000, step=1000)

print(f"Q3 (Chunking) - The total of chunks is {len(doc_chunks)}")

Q3 (Chunking) - The total of chunks is 295


In [5]:
chunk_texts = []
for chunk in doc_chunks:
    text = chunk["content"]
    chunk_texts.append(text)

def generate_embeddings(embedder_model, texts):
    embeddings = []
    batch_size = 50

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        batch_embeddings = embedder_model.encode_batch(batch)
        embeddings.extend(batch_embeddings)

    return np.array(embeddings)

corpus_embeddings = generate_embeddings(embedder, chunk_texts)

similarity_scores = corpus_embeddings.dot(nn_query_embedding)
best_score_idx = similarity_scores.argmax()
best_chunk = doc_chunks[best_score_idx]

print(f"The highest-scoring filename is {best_chunk['filename']}")

  0%|          | 0/6 [00:00<?, ?it/s]

The highest-scoring filename is 02-vector-search/lessons/07-sqlitesearch-vector.md


### Q4. Vector search with minsearch

In [6]:
from minsearch import VectorSearch

vector_index = VectorSearch(keyword_fields=["content"])
vector_index.fit(corpus_embeddings, doc_chunks)

eval_query = "What metric do we use to evaluate a search engine?"
eval_query_embedding = embedder.encode(eval_query)

vector_search_results = vector_index.search(eval_query_embedding)
best_vector_filename = vector_search_results[0]["filename"]

print(f"The filename is {best_vector_filename}")

The filename is 04-evaluation/lessons/05-search-metrics.md


### Q5. Text search vs vector search

In [7]:
from minsearch import Index

text_index = Index(text_fields=["content"])
text_index.fit(doc_chunks)

pg_query = "How do I store vectors in PostgreSQL?"
text_search_results = text_index.search(pg_query)
best_text_filename = text_search_results[0]["filename"]

print(f"The file name is {best_text_filename}")

The file name is 02-vector-search/lessons/02-embeddings.md


### Q6. Hybrid search

In [8]:
def reciprocal_rank_fusion(result_lists, k=60, num_results=5):
    rrf_scores = {}
    rrf_docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank)
            rrf_docs[key] = doc

    ranked_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    return [rrf_docs[key] for key in ranked_keys[:num_results]]

hybrid_query = "How do I give the model access to tools?"
hybrid_query_embedding = embedder.encode(hybrid_query)

hybrid_vector_results = vector_index.search(hybrid_query_embedding)
hybrid_text_results = text_index.search(hybrid_query)
final_hybrid_results = reciprocal_rank_fusion([hybrid_vector_results, hybrid_text_results])

print(f"The filename is {final_hybrid_results[0]['filename']}")

The filename is 01-agentic-rag/lessons/13-function-calling.md
